# O Poder das Imagens: Análise Visual de Trump vs AOC
## Como os Políticos Usam Fotos para Moldar Sua Mensagem

Uma imagem vale mais que mil palavras -- e nas redes sociais políticas, as imagens que os políticos escolhem compartilhar são uma ferramenta poderosa de persuasão e comunicação. Seja uma foto de um comício lotado, um retrato oficial ou um infográfico sobre políticas, cada imagem é uma escolha deliberada projetada para moldar como a audiência percebe o político e sua mensagem.

Neste notebook, analisamos a **estratégia visual** de Trump e AOC no Twitter/X. Perguntamos:
- **Com que frequência** cada político inclui imagens em seus tweets?
- **Como são as imagens?** São claras ou escuras? Quais cores dominam?
- **Tweets com imagens recebem mais atenção** do que tweets apenas com texto?
- **Quando** eles postam imagens? Existe um padrão estratégico de horário?
- **Quais tweets com imagens tiveram melhor desempenho?** O que podemos aprender com o conteúdo visual de melhor performance?

Não é necessário ter conhecimento de programação ou estatística. Cada conceito é explicado conforme aparece.

---

**Termos-chave que você verá neste notebook:**
- **Imagem / Foto**: Uma imagem anexada a um tweet (diferente de um tweet apenas com texto)
- **Média**: Some todos os números e divida pela quantidade deles
- **Mediana (valor do meio)**: Organize todos os números do menor para o maior; a mediana é aquele no meio
- **Taxa de engajamento**: Uma medida de quanta atenção um tweet recebe (curtidas, repostagens, respostas, etc.) em relação a quantas pessoas o viram
- **Distribuição**: Como os valores estão espalhados -- por exemplo, a maioria das imagens é clara, ou existe uma ampla variedade?

In [ ]:
# --- Configuração: Carregando as ferramentas e dados que precisamos ---
# (Esta célula carrega as bibliotecas de software e estilização. Você pode ignorar os detalhes técnicos.)

import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from pathlib import Path
from PIL import Image
import os

from utils.data_loader import load_tweets, get_images_dir
from utils.plot_helpers import (
    setup_style, get_colors, get_user_label,
    comparative_bar, format_large_numbers, set_language
)
from config import FIGURE_SIZE, FIGURE_SIZE_SMALL, FIGURE_SIZE_LARGE, RANDOM_SEED, TARGET_USERS

# Aplicar estilo consistente
setup_style()
set_language('pt-br')
colors = get_colors()
np.random.seed(RANDOM_SEED)

# Configurações de exibição
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Configuração concluída.')

In [ ]:
# --- Carregar dados e localizar imagens ---

df = load_tweets()
images_dir = get_images_dir()

print(f"Diretório de imagens: {images_dir}")
print()

# Construir inventário de imagens
image_inventory = {}
for user_label in ['trump', 'aoc']:
    user_dir = images_dir / user_label
    if user_dir.exists():
        files = sorted(user_dir.glob('*.jpg'))
        image_inventory[user_label] = files
        print(f"{user_label}: {len(files)} imagens encontradas em {user_dir}")
    else:
        image_inventory[user_label] = []
        print(f"{user_label}: Nenhum diretório de imagens encontrado em {user_dir}")

# Filtrar tweets com imagens
df_with_images = df[df['image_count'] > 0].copy()
df_without_images = df[df['image_count'] == 0].copy()

print(f"\nTweets com imagens: {len(df_with_images)} / {len(df)}")
for u in ['trump', 'aoc']:
    count = len(df_with_images[df_with_images['user'] == u])
    total = len(df[df['user'] == u])
    print(f"  {u}: {count} / {total} tweets com imagens ({count/total*100:.1f}%)")

## 1. Visão Geral do Uso de Imagens

Com que frequência cada político inclui imagens em seus tweets? A frequência e quantidade de conteúdo visual na comunicação política revelam escolhas estratégicas sobre como capturar atenção em feeds de redes sociais saturados. As imagens servem como "ganchos" visuais que podem aumentar o tempo que as pessoas passam olhando e a ressonância emocional.

In [ ]:
# --- Comparação de frequência de imagens ---

fig, axes = plt.subplots(1, 3, figsize=FIGURE_SIZE_LARGE)

users = ['trump', 'aoc']
user_labels = [get_user_label(u) for u in users]

# --- Painel 1: % de tweets contendo imagens ---
ax = axes[0]
pct_with_images = []
for u in users:
    user_total = len(df[df['user'] == u])
    user_with_img = len(df_with_images[df_with_images['user'] == u])
    pct_with_images.append(user_with_img / user_total * 100 if user_total > 0 else 0)

bars = ax.bar(user_labels, pct_with_images, color=[colors[u] for u in users],
              width=0.5, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, pct_with_images):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Tweets Contendo Imagens (%)', fontweight='bold')
ax.set_ylabel('Porcentagem')
ax.set_ylim(0, max(pct_with_images) * 1.2 if max(pct_with_images) > 0 else 10)

# --- Painel 2: Média de image_count por tweet ---
ax = axes[1]
avg_images = [df[df['user'] == u]['image_count'].mean() for u in users]
bars = ax.bar(user_labels, avg_images, color=[colors[u] for u in users],
              width=0.5, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, avg_images):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Média de Imagens por Tweet', fontweight='bold')
ax.set_ylabel('Média de Imagens')
ax.set_ylim(0, max(avg_images) * 1.3 if max(avg_images) > 0 else 1)

# --- Painel 3: Distribuição de tipo de mídia (gráficos de pizza) ---
axes[2].set_visible(False)
gs_inner = axes[2].get_subplotspec().subgridspec(1, 2, wspace=0.3)

pie_colors_map = {
    'none': '#CCCCCC',
    'photo': '#2A9D8F',
    'video': '#E9C46A',
    'animated_gif': '#F4A261',
    'mixed': '#E76F51'
}

for idx, user in enumerate(users):
    ax_pie = fig.add_subplot(gs_inner[0, idx])
    user_data = df[df['user'] == user]
    media_counts = user_data['media_type'].value_counts()
    
    # Obter cores para cada tipo de mídia
    pie_labels = media_counts.index.tolist()
    pie_vals = media_counts.values
    pie_colors = [pie_colors_map.get(str(label).lower(), '#999999') for label in pie_labels]
    
    wedges, texts, autotexts = ax_pie.pie(
        pie_vals, labels=None, autopct='%1.0f%%',
        colors=pie_colors, startangle=90,
        textprops={'fontsize': 8}
    )
    for autotext in autotexts:
        autotext.set_fontweight('bold')
    
    ax_pie.set_title(get_user_label(user).split(' (')[0], fontweight='bold', fontsize=11)
    
    # Adicionar legenda apenas para o segundo gráfico de pizza
    if idx == 1:
        ax_pie.legend(
            [str(l).title() for l in pie_labels],
            loc='center left', bbox_to_anchor=(1.0, 0.5),
            fontsize=8
        )

fig.suptitle('Visão Geral do Uso de Imagens', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Imprimir resumo
print("\nDetalhamento por tipo de mídia:")
for u in users:
    print(f"\n{get_user_label(u)}:")
    user_data = df[df['user'] == u]
    counts = user_data['media_type'].value_counts()
    for mt, cnt in counts.items():
        print(f"  {mt}: {cnt} ({cnt/len(user_data)*100:.1f}%)")

## 2. Galeria de Imagens

Uma visão geral visual das imagens usadas por cada político. Esta inspeção qualitativa é essencial -- antes de quantificar, devemos *observar*. Que tipos de imagens predominam? Retratos oficiais, fotos de comícios, infográficos, memes ou fotos pessoais? Cada tipo carrega um peso retórico distinto.

In [ ]:
# --- Grade de galeria de imagens ---

MAX_IMAGES = 20
GRID_COLS = 5
GRID_ROWS = 4

for user in ['trump', 'aoc']:
    user_images = image_inventory.get(user, [])
    if len(user_images) == 0:
        print(f"Nenhuma imagem disponível para {get_user_label(user)}")
        continue
    
    # Selecionar até MAX_IMAGES, igualmente espaçadas se existirem mais
    if len(user_images) > MAX_IMAGES:
        indices = np.linspace(0, len(user_images) - 1, MAX_IMAGES, dtype=int)
        selected = [user_images[i] for i in indices]
    else:
        selected = user_images[:MAX_IMAGES]
    
    n_images = len(selected)
    n_rows = int(np.ceil(n_images / GRID_COLS))
    
    fig, axes = plt.subplots(n_rows, GRID_COLS, figsize=(16, n_rows * 3.2))
    fig.suptitle(f'Galeria de Imagens: {get_user_label(user)}',
                 fontweight='bold', fontsize=14, y=1.01)
    
    # Achatar axes para indexação fácil
    if n_rows == 1:
        axes_flat = axes if isinstance(axes, np.ndarray) else [axes]
    else:
        axes_flat = axes.flatten()
    
    for i, ax in enumerate(axes_flat):
        if i < n_images:
            try:
                img = Image.open(selected[i])
                ax.imshow(img)
                # Extrair tweet_id do nome do arquivo (formato: {tweet_id}_{index}.jpg)
                fname = selected[i].stem
                ax.set_title(fname, fontsize=7, color='gray')
            except Exception as e:
                ax.text(0.5, 0.5, f'Erro:\n{str(e)[:30]}',
                        ha='center', va='center', fontsize=8,
                        transform=ax.transAxes, color='red')
                ax.set_facecolor('#f0f0f0')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    print(f"  Exibindo {n_images} de {len(user_images)} imagens no total\n")

## 3. Características das Imagens

Analisar as propriedades físicas das imagens revela escolhas de produção. Dimensões e proporções indicam o tipo de conteúdo (por exemplo, fotos paisagem, capturas de tela verticais, gráficos quadrados). Brilho e propriedades de cor podem sugerir o tom visual -- paletas quentes vs. frias, estéticas de alto contraste vs. baixo contraste.

In [ ]:
# --- Análise de imagens: dimensões, proporções, brilho, cor ---

image_stats = []

for user in ['trump', 'aoc']:
    user_images = image_inventory.get(user, [])
    for img_path in user_images:
        try:
            img = Image.open(img_path)
            width, height = img.size
            aspect_ratio = width / height if height > 0 else 0
            
            # Converter para RGB se necessário (lida com modos RGBA, P, L)
            img_rgb = img.convert('RGB')
            pixels = np.array(img_rgb, dtype=np.float64)
            
            # Brilho médio (média em escala de cinza / 255)
            gray = np.array(img.convert('L'), dtype=np.float64)
            brightness = gray.mean() / 255.0
            
            # Médias por canal
            avg_red = pixels[:, :, 0].mean()
            avg_green = pixels[:, :, 1].mean()
            avg_blue = pixels[:, :, 2].mean()
            
            # Canal de cor dominante
            channel_means = {'R': avg_red, 'G': avg_green, 'B': avg_blue}
            dominant_channel = max(channel_means, key=channel_means.get)
            
            # Saturação de cor (desvio padrão de todos os valores de pixel)
            color_saturation = pixels.std()
            
            # Extrair tweet_id do nome do arquivo
            fname = img_path.stem
            tweet_id = fname.rsplit('_', 1)[0] if '_' in fname else fname
            
            image_stats.append({
                'user': user,
                'filename': img_path.name,
                'tweet_id': tweet_id,
                'width': width,
                'height': height,
                'aspect_ratio': aspect_ratio,
                'brightness': brightness,
                'dominant_channel': dominant_channel,
                'avg_red': avg_red,
                'avg_green': avg_green,
                'avg_blue': avg_blue,
                'color_saturation': color_saturation
            })
            img.close()
        except Exception as e:
            print(f"  Pulando {img_path.name}: {e}")

df_images = pd.DataFrame(image_stats)

if len(df_images) > 0:
    print(f"Analisadas com sucesso {len(df_images)} imagens.")
    print(f"\nDetalhamento por usuário:")
    for u in ['trump', 'aoc']:
        count = len(df_images[df_images['user'] == u])
        print(f"  {get_user_label(u)}: {count} imagens")
    print(f"\nResumo das propriedades das imagens:")
    display(df_images.groupby('user')[['width', 'height', 'aspect_ratio', 'brightness', 'color_saturation']].describe().round(2).T)
else:
    print("Nenhuma imagem pôde ser analisada.")

In [ ]:
# --- Análise de dimensões e proporções ---

if len(df_images) > 0:
    fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE_LARGE)
    
    # --- Painel 1: Histograma de proporções ---
    ax = axes[0]
    for user in ['trump', 'aoc']:
        user_data = df_images[df_images['user'] == user]['aspect_ratio'].dropna()
        ax.hist(user_data, bins=20, alpha=0.55, color=colors[user],
                edgecolor='white', linewidth=0.5, label=get_user_label(user))
        mean_val = user_data.mean()
        ax.axvline(mean_val, color=colors[user], linestyle='--', linewidth=2, alpha=0.8)
        ax.text(mean_val, ax.get_ylim()[1] * 0.9, f'  {mean_val:.2f}',
                color=colors[user], fontsize=9, fontweight='bold')
    ax.set_title('Distribuição da Proporção das Imagens', fontweight='bold')
    ax.set_xlabel('Proporção (largura / altura)')
    ax.set_ylabel('Frequência')
    ax.legend()
    
    # --- Painel 2: Dispersão largura vs altura ---
    ax = axes[1]
    for user in ['trump', 'aoc']:
        user_data = df_images[df_images['user'] == user]
        ax.scatter(user_data['width'], user_data['height'],
                   color=colors[user], alpha=0.5, s=40, edgecolor='white',
                   linewidth=0.5, label=get_user_label(user))
    # Adicionar linha diagonal para proporção 1:1
    max_dim = max(df_images['width'].max(), df_images['height'].max())
    ax.plot([0, max_dim], [0, max_dim], color='gray', linestyle=':', linewidth=1, alpha=0.5)
    ax.set_title('Dimensões das Imagens (Largura vs Altura)', fontweight='bold')
    ax.set_xlabel('Largura (px)')
    ax.set_ylabel('Altura (px)')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Imprimir proporções mais comuns
    print("\nProporções mais comuns (arredondadas para 1 casa decimal):")
    for u in ['trump', 'aoc']:
        user_data = df_images[df_images['user'] == u]
        rounded = user_data['aspect_ratio'].round(1)
        top = rounded.value_counts().head(3)
        print(f"  {get_user_label(u)}:")
        for ratio, cnt in top.items():
            print(f"    {ratio:.1f} - {cnt} imagens")
else:
    print("Nenhum dado de imagem disponível para análise de dimensões.")

In [ ]:
# --- Análise de brilho ---

if len(df_images) > 0:
    fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE)
    
    # --- Painel 1: Histograma de brilho ---
    ax = axes[0]
    for user in ['trump', 'aoc']:
        user_data = df_images[df_images['user'] == user]['brightness'].dropna()
        ax.hist(user_data, bins=20, alpha=0.55, color=colors[user],
                edgecolor='white', linewidth=0.5, label=get_user_label(user))
        mean_val = user_data.mean()
        ax.axvline(mean_val, color=colors[user], linestyle='--', linewidth=2, alpha=0.8)
    ax.set_title('Distribuição de Brilho das Imagens', fontweight='bold')
    ax.set_xlabel('Brilho (0=preto, 1=branco)')
    ax.set_ylabel('Frequência')
    ax.legend()
    
    # --- Painel 2: Brilho médio ---
    ax = axes[1]
    mean_brightness = [df_images[df_images['user'] == u]['brightness'].mean() for u in users]
    user_labels_short = [get_user_label(u) for u in users]
    bars = ax.bar(user_labels_short, mean_brightness, color=[colors[u] for u in users],
                  width=0.5, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, mean_brightness):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_title('Brilho Médio das Imagens', fontweight='bold')
    ax.set_ylabel('Brilho (0-1)')
    ax.set_ylim(0, min(max(mean_brightness) * 1.3, 1.0))
    
    plt.tight_layout()
    plt.show()
    
    # Interpretação
    for u in users:
        b = df_images[df_images['user'] == u]['brightness']
        print(f"{get_user_label(u)} brilho: média={b.mean():.3f}, mediana={b.median():.3f}, desvio={b.std():.3f}")
else:
    print("Nenhum dado de imagem disponível para análise de brilho.")

## 4. Análise de Paleta de Cores

Comparando o "tom" visual das imagens usadas por cada político. A cor é uma ferramenta retórica poderosa: tons quentes (vermelhos, laranjas) evocam energia e urgência, enquanto tons frios (azuis, verdes) sugerem calma e autoridade. O perfil médio de cores das imagens de um político contribui para sua **identidade de marca visual** geral.

In [ ]:
# --- Cor média por usuário ---

if len(df_images) > 0:
    fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE)
    
    # --- Painel 1: Amostras de cor mostrando a cor média ---
    ax = axes[0]
    avg_colors_per_user = {}
    for idx, user in enumerate(users):
        user_data = df_images[df_images['user'] == user]
        avg_r = user_data['avg_red'].mean() / 255.0
        avg_g = user_data['avg_green'].mean() / 255.0
        avg_b = user_data['avg_blue'].mean() / 255.0
        avg_colors_per_user[user] = (avg_r, avg_g, avg_b)
        
        # Desenhar retângulo colorido
        rect = plt.Rectangle((idx * 1.5, 0), 1.0, 1.0,
                             facecolor=(avg_r, avg_g, avg_b), edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        
        # Rótulo abaixo
        hex_color = '#{:02x}{:02x}{:02x}'.format(
            int(avg_r * 255), int(avg_g * 255), int(avg_b * 255))
        ax.text(idx * 1.5 + 0.5, -0.15, get_user_label(user).split(' (')[0],
                ha='center', va='top', fontsize=11, fontweight='bold')
        ax.text(idx * 1.5 + 0.5, -0.30, hex_color,
                ha='center', va='top', fontsize=10, color='gray')
        ax.text(idx * 1.5 + 0.5, 0.5,
                f'R:{avg_r*255:.0f}\nG:{avg_g*255:.0f}\nB:{avg_b*255:.0f}',
                ha='center', va='center', fontsize=10, fontweight='bold',
                color='white' if (avg_r + avg_g + avg_b) / 3 < 0.5 else 'black')
    
    ax.set_xlim(-0.3, len(users) * 1.5)
    ax.set_ylim(-0.5, 1.3)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('Cor Média das Imagens', fontweight='bold', fontsize=13)
    
    # --- Painel 2: Barras agrupadas de R, G, B médio por usuário ---
    ax = axes[1]
    x = np.arange(3)  # R, G, B
    bar_width = 0.3
    channel_labels = ['Vermelho', 'Verde', 'Azul']
    channel_colors_bar = ['#E63946', '#2A9D8F', '#457B9D']
    
    for i, user in enumerate(users):
        user_data = df_images[df_images['user'] == user]
        means = [
            user_data['avg_red'].mean(),
            user_data['avg_green'].mean(),
            user_data['avg_blue'].mean()
        ]
        offset = -bar_width / 2 + i * bar_width
        bars = ax.bar(x + offset, means, width=bar_width,
                      color=colors[user], alpha=0.85,
                      label=get_user_label(user),
                      edgecolor='white', linewidth=0.5)
        for bar, val in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    f'{val:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax.set_title('Canais de Cor Médios', fontweight='bold', fontsize=13)
    ax.set_xlabel('Canal de Cor')
    ax.set_ylabel('Valor Médio de Pixel (0-255)')
    ax.set_xticks(x)
    ax.set_xticklabels(channel_labels)
    ax.set_ylim(0, 280)
    ax.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum dado de imagem disponível para análise de cores.")

In [ ]:
# --- Distribuição do canal de cor dominante ---

if len(df_images) > 0:
    fig, ax = plt.subplots(figsize=FIGURE_SIZE_SMALL)
    
    channel_order = ['R', 'G', 'B']
    channel_display = ['Vermelho', 'Verde', 'Azul']
    channel_bar_colors = ['#E63946', '#2A9D8F', '#457B9D']
    
    x = np.arange(len(users))
    bottom = np.zeros(len(users))
    
    for ch_idx, channel in enumerate(channel_order):
        pcts = []
        for user in users:
            user_data = df_images[df_images['user'] == user]
            total = len(user_data)
            ch_count = (user_data['dominant_channel'] == channel).sum()
            pcts.append(ch_count / total * 100 if total > 0 else 0)
        
        bars = ax.bar(x, pcts, bottom=bottom, width=0.5,
                      color=channel_bar_colors[ch_idx], alpha=0.85,
                      label=channel_display[ch_idx],
                      edgecolor='white', linewidth=0.5)
        
        # Adicionar rótulos em segmentos grandes o suficiente
        for i, (bar, pct) in enumerate(zip(bars, pcts)):
            if pct >= 5:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bottom[i] + pct / 2,
                        f'{pct:.0f}%', ha='center', va='center',
                        fontsize=10, fontweight='bold', color='white')
        bottom += pcts
    
    ax.set_title('Distribuição do Canal de Cor Dominante', fontweight='bold', fontsize=13)
    ax.set_ylabel('Porcentagem de Imagens (%)')
    ax.set_xticks(x)
    ax.set_xticklabels([get_user_label(u) for u in users])
    ax.set_ylim(0, 110)
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()
    
    # Imprimir valores exatos
    print("\nDetalhamento do canal dominante:")
    for u in users:
        user_data = df_images[df_images['user'] == u]
        total = len(user_data)
        print(f"  {get_user_label(u)}:")
        for ch in channel_order:
            cnt = (user_data['dominant_channel'] == ch).sum()
            print(f"    {ch}: {cnt} ({cnt/total*100:.1f}%)")
else:
    print("Nenhum dado de imagem disponível para análise de cor dominante.")

## 5. Correlação Imagem vs Engajamento

Tweets com imagens geram mais engajamento? Esta é uma questão central para entender o valor estratégico do conteúdo visual. Se as imagens consistentemente impulsionam o engajamento, isso revela a preferência da audiência por comunicação visual e pode explicar a estratégia de mídia de cada político.

In [ ]:
# --- Comparação de engajamento com imagens ---

fig, axes = plt.subplots(1, 3, figsize=FIGURE_SIZE_LARGE)

# Adicionar flag 'has_image'
df['has_image'] = df['image_count'] > 0

# --- Painel 1: Média de engagement_rate para imagem vs sem imagem ---
ax = axes[0]
x = np.arange(len(users))
bar_width = 0.3
group_colors = ['#ADB5BD', '#2A9D8F']
group_labels_display = ['Sem Imagens', 'Com Imagens']

for g_idx, has_img in enumerate([False, True]):
    means = []
    for user in users:
        user_data = df[(df['user'] == user) & (df['has_image'] == has_img)]
        means.append(user_data['engagement_rate'].mean() if len(user_data) > 0 else 0)
    offset = -bar_width / 2 + g_idx * bar_width
    bars = ax.bar(x + offset, means, width=bar_width,
                  color=group_colors[g_idx], alpha=0.85,
                  label=group_labels_display[g_idx],
                  edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, means):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    f'{val:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('Taxa de Engajamento Média', fontweight='bold')
ax.set_ylabel('Taxa de Engajamento (%)')
ax.set_xticks(x)
ax.set_xticklabels([get_user_label(u).split(' (')[0] for u in users])
ax.legend(fontsize=9)

# --- Painel 2: Box plot de engagement_rate ---
ax = axes[1]
box_data = []
box_labels = []
box_colors_list = []
for user in users:
    for has_img, label_suffix in [(False, 'S/ Img'), (True, 'Img')]:
        subset = df[(df['user'] == user) & (df['has_image'] == has_img)]['engagement_rate'].dropna()
        if len(subset) > 0:
            box_data.append(subset.values)
            short_name = 'T' if user == 'trump' else 'A'
            box_labels.append(f'{short_name}-{label_suffix}')
            box_colors_list.append(colors[user])

if box_data:
    bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True, widths=0.5,
                    medianprops={'color': 'black', 'linewidth': 1.5})
    for patch, c in zip(bp['boxes'], box_colors_list):
        patch.set_facecolor(c)
        patch.set_alpha(0.6)

ax.set_title('Distribuição da Taxa de Engajamento', fontweight='bold')
ax.set_ylabel('Taxa de Engajamento (%)')
ax.tick_params(axis='x', rotation=30)

# --- Painel 3: Média de curtidas ---
ax = axes[2]
for g_idx, has_img in enumerate([False, True]):
    means = []
    for user in users:
        user_data = df[(df['user'] == user) & (df['has_image'] == has_img)]
        means.append(user_data['favorite_count'].mean() if len(user_data) > 0 else 0)
    offset = -bar_width / 2 + g_idx * bar_width
    bars = ax.bar(x + offset, means, width=bar_width,
                  color=group_colors[g_idx], alpha=0.85,
                  label=group_labels_display[g_idx],
                  edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, means):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    f'{val:,.0f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('Média de Curtidas', fontweight='bold')
ax.set_ylabel('Média de Curtidas')
ax.set_xticks(x)
ax.set_xticklabels([get_user_label(u).split(' (')[0] for u in users])
ax.legend(fontsize=9)
format_large_numbers(ax)

fig.suptitle('Imagens Impulsionam o Engajamento?', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Imprimir tabela resumo
print("\nComparação de engajamento (tweets com imagem vs sem imagem):")
print(f"{'Usuário':<12} {'Grupo':<16} {'Tweets':>8} {'Taxa Eng. Média':>16} {'Média Curtidas':>14}")
print('-' * 68)
for user in users:
    label = get_user_label(user).split(' (')[0]
    for has_img, group_name in [(False, 'Sem Imagens'), (True, 'Com Imagens')]:
        subset = df[(df['user'] == user) & (df['has_image'] == has_img)]
        if len(subset) > 0:
            print(f"{label:<12} {group_name:<16} {len(subset):>8} "
                  f"{subset['engagement_rate'].mean():>15.3f}% "
                  f"{subset['favorite_count'].mean():>13,.0f}")

## 6. Padrões de Postagem Visual

Quando eles postam imagens? Os padrões temporais de conteúdo visual podem diferir dos padrões gerais de postagem. Por exemplo, um político pode postar comentários com muito texto durante o horário comercial, mas compartilhar imagens nos horários de pico da audiência para máximo impacto visual.

In [ ]:
# --- Padrões temporais de postagem de imagens ---

fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE_LARGE)

# --- Painel 1: Distribuição por hora do dia ---
ax = axes[0]
x_hours = np.arange(24)
bar_width = 0.35

for i, user in enumerate(users):
    user_img_tweets = df_with_images[df_with_images['user'] == user]
    hour_counts = user_img_tweets['hour'].value_counts().reindex(range(24), fill_value=0)
    offset = -bar_width / 2 + i * bar_width
    ax.bar(x_hours + offset, hour_counts.values,
           width=bar_width, color=colors[user], alpha=0.85,
           label=get_user_label(user), edgecolor='white', linewidth=0.3)

ax.set_title('Tweets com Imagens por Hora do Dia (UTC)', fontweight='bold')
ax.set_xlabel('Hora do Dia')
ax.set_ylabel('Número de Tweets com Imagens')
ax.set_xticks(range(0, 24, 3))
ax.set_xlim(-0.5, 23.5)
ax.legend()

# --- Painel 2: Distribuição por dia da semana ---
ax = axes[1]
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_abbr = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
x_days = np.arange(len(day_order))
bar_width = 0.35

for i, user in enumerate(users):
    user_img_tweets = df_with_images[df_with_images['user'] == user]
    day_counts = user_img_tweets['day_name'].value_counts().reindex(day_order, fill_value=0)
    offset = -bar_width / 2 + i * bar_width
    bars = ax.bar(x_days + offset, day_counts.values,
                  width=bar_width, color=colors[user], alpha=0.85,
                  label=get_user_label(user), edgecolor='white', linewidth=0.3)
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, height,
                    f'{int(height)}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('Tweets com Imagens por Dia da Semana', fontweight='bold')
ax.set_xlabel('Dia da Semana')
ax.set_ylabel('Número de Tweets com Imagens')
ax.set_xticks(x_days)
ax.set_xticklabels(day_abbr)
ax.legend()

plt.tight_layout()
plt.show()

# Imprimir horários de pico
print("\nHorários de pico de postagem de imagens (UTC):")
for u in users:
    user_img = df_with_images[df_with_images['user'] == u]
    if len(user_img) > 0:
        peak_hour = user_img['hour'].mode().iloc[0] if len(user_img['hour'].mode()) > 0 else 'N/A'
        print(f"  {get_user_label(u)}: {peak_hour}:00 UTC")

## 7. Tweets com Imagens de Melhor Desempenho

Os tweets mais engajadores que incluem imagens. Ao examinar essas postagens visuais de alto desempenho, podemos identificar o que faz certas imagens ressoarem com as audiências.

In [ ]:
# --- Top 5 tweets com imagens por engajamento ---

TOP_N = 5

for user in ['trump', 'aoc']:
    user_img_tweets = df_with_images[df_with_images['user'] == user].copy()
    
    if len(user_img_tweets) == 0:
        print(f"Nenhum tweet com imagem disponível para {get_user_label(user)}")
        continue
    
    top_tweets = user_img_tweets.nlargest(TOP_N, 'total_engagement')
    n_show = len(top_tweets)
    
    fig, axes = plt.subplots(n_show, 2, figsize=(14, n_show * 3.5),
                             gridspec_kw={'width_ratios': [1, 2]})
    fig.suptitle(f'Top {n_show} Tweets com Imagens: {get_user_label(user)}',
                 fontweight='bold', fontsize=14, y=1.01)
    
    if n_show == 1:
        axes = np.array([axes])
    
    for idx, (_, row) in enumerate(top_tweets.iterrows()):
        ax_img = axes[idx, 0]
        ax_text = axes[idx, 1]
        
        # Tentar carregar a primeira imagem deste tweet
        image_loaded = False
        if pd.notna(row.get('image_paths', None)) and str(row['image_paths']).strip():
            paths = str(row['image_paths']).split(',')
            for img_path_str in paths:
                img_path_str = img_path_str.strip()
                if img_path_str:
                    try:
                        img = Image.open(img_path_str)
                        ax_img.imshow(img)
                        image_loaded = True
                        img.close()
                        break
                    except Exception:
                        pass
        
        # Fallback: buscar no inventário pelo tweet_id
        if not image_loaded:
            tweet_id_str = str(row['tweet_id'])
            for inv_path in image_inventory.get(user, []):
                if inv_path.stem.startswith(tweet_id_str):
                    try:
                        img = Image.open(inv_path)
                        ax_img.imshow(img)
                        image_loaded = True
                        img.close()
                        break
                    except Exception:
                        pass
        
        if not image_loaded:
            ax_img.text(0.5, 0.5, 'Imagem\nIndisponível',
                        ha='center', va='center', fontsize=12,
                        color='gray', transform=ax_img.transAxes)
            ax_img.set_facecolor('#f5f5f5')
        
        ax_img.axis('off')
        
        # Painel de texto e métricas
        text_content = str(row.get('text', ''))[:200]
        if len(str(row.get('text', ''))) > 200:
            text_content += '...'
        
        metrics_str = (
            f"Curtidas: {row['favorite_count']:,.0f}  |  "
            f"Repostagens: {row['retweet_count']:,.0f}  |  "
            f"Respostas: {row['reply_count']:,.0f}\n"
            f"Visualizações: {row['view_count']:,.0f}  |  "
            f"Taxa de Engajamento: {row['engagement_rate']:.2f}%\n"
            f"Engajamento Total: {row['total_engagement']:,.0f}"
        )
        
        ax_text.text(0.02, 0.95, f"#{idx+1}", fontsize=14, fontweight='bold',
                     color=colors[user], transform=ax_text.transAxes,
                     va='top')
        ax_text.text(0.02, 0.78, text_content, fontsize=9,
                     transform=ax_text.transAxes, va='top',
                     wrap=True, family='monospace')
        ax_text.text(0.02, 0.15, metrics_str, fontsize=9,
                     transform=ax_text.transAxes, va='top',
                     fontweight='bold', color='#333333',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='#f0f0f0', alpha=0.8))
        ax_text.axis('off')
    
    plt.tight_layout()
    plt.show()
    print()

## 8. Síntese da Identidade Visual

Como a estratégia visual de cada político sustenta sua identidade digital mais ampla? Esta tabela resumo consolida as principais métricas visuais para comparação direta.

In [ ]:
# --- Tabela comparativa resumo ---

summary_rows = []

for user in users:
    user_all = df[df['user'] == user]
    user_img = df_with_images[df_with_images['user'] == user]
    user_no_img = df_without_images[df_without_images['user'] == user]
    user_img_props = df_images[df_images['user'] == user] if len(df_images) > 0 else pd.DataFrame()
    
    total = len(user_all)
    pct_images = len(user_img) / total * 100 if total > 0 else 0
    avg_img_per_tweet = user_all['image_count'].mean() if total > 0 else 0
    
    # Propriedades das imagens
    mean_brightness = user_img_props['brightness'].mean() if len(user_img_props) > 0 else np.nan
    
    # Cor média como hex
    if len(user_img_props) > 0:
        avg_r = int(user_img_props['avg_red'].mean())
        avg_g = int(user_img_props['avg_green'].mean())
        avg_b = int(user_img_props['avg_blue'].mean())
        avg_hex = f'#{avg_r:02x}{avg_g:02x}{avg_b:02x}'
    else:
        avg_hex = 'N/A'
    
    # Taxas de engajamento
    eng_rate_img = user_img['engagement_rate'].mean() if len(user_img) > 0 else np.nan
    eng_rate_no_img = user_no_img['engagement_rate'].mean() if len(user_no_img) > 0 else np.nan
    
    # Impulso de engajamento
    if eng_rate_no_img > 0 and not np.isnan(eng_rate_img) and not np.isnan(eng_rate_no_img):
        eng_boost = ((eng_rate_img - eng_rate_no_img) / eng_rate_no_img) * 100
    else:
        eng_boost = np.nan
    
    # Hora mais comum para tweets com imagem
    if len(user_img) > 0 and len(user_img['hour'].mode()) > 0:
        peak_hour = f"{user_img['hour'].mode().iloc[0]}:00 UTC"
    else:
        peak_hour = 'N/A'
    
    summary_rows.append({
        'Métrica': get_user_label(user),
        '% Tweets com Imagens': f'{pct_images:.1f}%',
        'Média Imagens por Tweet': f'{avg_img_per_tweet:.2f}',
        'Brilho Médio': f'{mean_brightness:.3f}' if not np.isnan(mean_brightness) else 'N/A',
        'Cor Média (hex)': avg_hex,
        'Taxa Eng. (Imagem)': f'{eng_rate_img:.3f}%' if not np.isnan(eng_rate_img) else 'N/A',
        'Taxa Eng. (Sem Imagem)': f'{eng_rate_no_img:.3f}%' if not np.isnan(eng_rate_no_img) else 'N/A',
        'Impulso Eng. Imagem': f'{eng_boost:+.1f}%' if not np.isnan(eng_boost) else 'N/A',
        'Hora Pico Imagem': peak_hour
    })

summary_df = pd.DataFrame(summary_rows).set_index('Métrica').T
summary_df.index.name = 'Métrica Visual'

# Estilizar a tabela
def style_visual_summary(styler):
    styler.set_caption('Comparação de Identidade Visual: Trump vs AOC')
    styler.set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold'), ('margin-bottom', '10px')]},
        {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('font-weight', 'bold'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '8px')]},
    ])
    # Colorir colunas
    trump_col = [c for c in summary_df.columns if 'trump' in c.lower() or 'Trump' in c]
    aoc_col = [c for c in summary_df.columns if 'aoc' in c.lower() or 'AOC' in c]
    if trump_col:
        styler.set_properties(subset=trump_col, **{'background-color': '#fce4e4'})
    if aoc_col:
        styler.set_properties(subset=aoc_col, **{'background-color': '#e4ecf4'})
    return styler

display(summary_df.style.pipe(style_visual_summary))

# Também imprimir em texto simples
print("\nResumo em texto simples:")
print(summary_df.to_string())

## 9. Principais Conclusões

Esta análise visual revela abordagens distintas de **retórica visual** na comunicação política digital:

### Estratégia de Uso de Imagens
- A frequência com que cada político inclui imagens em seus tweets reflete abordagens fundamentalmente diferentes de comunicação visual. Essas escolhas moldam como as audiências encontram e processam suas mensagens em feeds curados algoritmicamente.

### Estética Visual
- Brilho, paletas de cores e dimensões das imagens não são acidentais -- formam uma **identidade visual** coerente. Diferenças no brilho médio podem refletir tons retóricos contrastantes (otimista de alto contraste vs. dramático de baixo contraste), enquanto perfis de cores reforçam o reconhecimento da marca.

### Impacto no Engajamento
- A relação entre conteúdo visual e métricas de engajamento revela se as audiências respondem de forma diferente a tweets somente texto vs. tweets com imagens. Qualquer "impulso de engajamento" das imagens quantifica o valor estratégico do conteúdo visual para cada político.

### Estratégia Visual Temporal
- O momento das postagens com imagens pode diferir dos padrões gerais de postagem, sugerindo agendamento deliberado de conteúdo visual para máximo impacto na audiência.

### Nota Metodológica
Esta análise combina **processamento quantitativo de imagens** (análise de brilho e cor no nível de pixel via PIL) com **inspeção visual qualitativa** (galerias de imagens e tweets de melhor desempenho). Esta abordagem de métodos mistos alinha-se com o framework netnográfico, que enfatiza a integração de observação sistemática com análise cultural interpretativa.

---

*Este notebook encerra a série de análises, integrando descobertas visuais ao retrato netnográfico abrangente das estratégias de comunicação política digital de Trump e AOC.*